In [1]:
import json
import pandas as pd
import re

import requests
import zipfile
import io
# Extract the contents of the zip file to a temporary directory
import tempfile
import os
import shutil

In [2]:
# Download the zip file from the URL
temp_dir = tempfile.mkdtemp()

def download_unzip_files(temp_dir):
    url = 'https://owncloud.gwdg.de/index.php/s/wqUJmN53BYZK9ff/download'
    response = requests.get(url)
    z = zipfile.ZipFile(io.BytesIO(response.content))

    
    z.extractall(temp_dir)

def delete_temporary_files(temp_dir):
    shutil.rmtree(temp_dir)


In [4]:
# Opening JSON file

def remove_blanks_concat(txt_list):
    clean_list = [x for x in txt_list if x!="_BLANK"]
    concat = " ".join(clean_list)
    
    return concat

def remove_double_spaces(txt):
    clean_txt = re.sub(' +', ' ', txt).strip()
    return clean_txt


# Process json files
download_unzip_files(temp_dir)


dict_data={}
list_df_labels=[]

for root, dirs, files in os.walk(temp_dir):
    for filename in files:
        # Do something with each file
        file_path = os.path.join(root, filename)
        category= filename.split("_")[1]
        if "json" in filename:
            f = open(file_path)
            data = json.load(f)
            dict_data[category] = data

        if "csv" in filename:
            df_label = pd.read_csv(file_path)
            df_label["split"] = category
            list_df_labels.append(df_label)


# turn headlines into table
list_dfs= []
for k,v in dict_data.items():
    df= pd.DataFrame.from_dict(v , orient='index')
    df["split"] = k
    list_dfs.append(df)


df = pd.concat(list_dfs)
df_labels = pd.concat(list_df_labels)

df["text"]= df["text"].apply(remove_blanks_concat)
df["text"]= df["text"].apply(remove_double_spaces)
df["headline"]= df["headline"].apply(remove_double_spaces)

print(df.split.value_counts())

BadZipFile: File is not a zip file

In [4]:
df_labels.split.value_counts()

split
test     30342
train     3378
valid      678
Name: count, dtype: int64

In [5]:
# delete_temporary_files(temp_dir)

In [5]:
df.query("headline!=text").head()

,mid,midid,incidentid,headline,text,date,split
34391NYT,False,NaN,34391NYTIncident,World Briefing | Europe: Britain: New Vote For...,World Briefing Europe Britain New Vote For Tor...,2001-07-11,test
37076NYT,False,NaN,37076NYTIncident,A NATION CHALLENGED: THE VICTIMS; In Village W...,A NATION CHALLENGED THE VICTIMS In Village Whe...,2001-12-16,test
25010NYT,False,NaN,25010NYTIncident,C.I.A. Study Details Failures; Scouring of Sys...,C I A Study Details Failures Scouring of Syste...,1998-06-03,test
33850NYT,False,NaN,33850NYTIncident,David Spedding Dies at 58; Headed British Spy ...,David Spedding Dies at 58 Headed British Spy A...,2001-06-16,test
29011NYT,False,NaN,29011NYTIncident,CRISIS IN THE BALKANS: ON THE GROUND; Rebel Of...,CRISIS IN THE BALKANS ON THE GROUND Rebel Offi...,1999-04-04,test


### Will use headline column to keep punctuation


In [6]:
df.query('midid=="3552"')

,mid,midid,incidentid,headline,text,date,split
007MID,True,3552,3552,Item 0727029 load date July 28 1992,Item 0727029 load date July 28 1992,1992-07-07,train
008MID,True,3552,3552,Bush bolsters military in Gulf,Bush bolsters military in Gulf,1992-07-07,train
009MID,True,3552,3552,Bush gets bipartisan support on Iraq policy,Bush gets bipartisan support on Iraq policy,1992-07-07,train
011MID,True,3552,3552004,Iraq condemns proposed no fly zone as imperial...,Iraq condemns proposed no fly zone as imperial...,1992-08-21,train
012MID,True,3552,3552004,Iraq says it is ready to fight allies over air...,Iraq says it is ready to fight allies over air...,1992-08-27,train
013MID,True,3552,3552004,US and allies say flight ban in Iraq will star...,US and allies say flight ban in Iraq will star...,1992-08-27,train
015MID,True,3552,3552003,US conducts three Gulf military exercises in s...,US conducts three Gulf military exercises in s...,1992-08-03,train
016MID,True,3552,3552003,Earger Amce military exercises beginning in th...,Earger Amce military exercises beginning in th...,1992-08-03,train


In [7]:
df.query('incidentid=="4122001"')

,mid,midid,incidentid,headline,text,date,split
134MID,True,4122,4122001,heavy fightings reported on zairean rwandan bo...,heavy fightings reported on zairean rwandan bo...,1996-09-23,train
135MID,True,4122,4122001,Zairian Troops Fire Across Rwandan Border,Zairian Troops Fire Across Rwandan Border,1996-09-23,train
136MID,True,4122,4122001,zaire rwanda agree to observe border ceasefire,zaire rwanda agree to observe border ceasefire,1996-09-24,train
137MID,True,4122,4122001,rwanda zaire exchange fire overnight,rwanda zaire exchange fire overnight,1996-09-24,train
138MID,True,4122,4122001,Fresh fighting on border between Rwanda and Zaire,Fresh fighting on border between Rwanda and Zaire,1996-09-24,train
139MID,True,4122,4122001,Zairian Rwandan Troops Battle for Second Night,Zairian Rwandan Troops Battle for Second Night,1996-09-24,train
140MID,True,4122,4122001,Zaire and Rwanda in border clashes,Zaire and Rwanda in border clashes,1996-09-25,train
141MID,True,4122,4122001,Rwanda and Zaire agree ceasefire after border ...,Rwanda and Zaire agree ceasefire after border ...,1996-09-25,train
142MID,True,4122,4122001,Tension with Zaire,Tension with Zaire,1996-09-19,train


In [10]:
df.to_csv('df_features.csv')
df_labels.to_csv('df_labels.csv')

In [11]:
# Export files
df.reset_index().to_feather("df_features.fth")
df_labels.reset_index().to_feather('df_labels.fth')